# Step 1 — what can we tune in Random Forest?

Random Forest builds many decision trees on random subsets of data and features. The tuning controls: how many trees, how deep each tree, and how many features each tree sees.

In [ ]:
# RF Hyperparameters:
# n_estimators   — number of trees. More = better (diminishing returns after ~300)
# max_depth      — max depth per tree. None is often fine for RF
# max_features   — how many features each tree considers for splits
# min_samples_split — minimum samples to split a node

# Why RF doesn't overfit as badly as a single tree:
# Each tree sees different random rows (bootstrap sample)
# Each split considers only sqrt(n_features) features
# Averaging N trees cancels out individual tree noise

In [ ]:
n_estimators=100 ---> 100 trees. Good baseline. After 300-500, improvement is marginal but compute increases

max_features='sqrt' ---> Each split looks at sqrt(n_features) random features. Default for classification

max_features='log2' ---> log2(n_features) features per split. More aggressive randomness

max_features=1.0 ---> All features at every split — this makes it a bagged tree, not random forest

max_depth=None ---> Unlike single DT, letting trees grow fully is often fine in RF due to averaging

bootstrap=True ---> Each tree trained on a random sample with replacement. Default and recommended

RF is already a strong baseline model. Don't over-tune it. Start with n_estimators=200, max_features='sqrt' and see if you even need more tuning.

Step 2 — prepare data

No scaling needed. RF is tree-based. Just split the data.

In [1]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, RandomizedSearchCV
import numpy as np
import pandas as pd

X, y = load_breast_cancer(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# No scaling! RF does not need it.
# Check baseline before tuning:
base_rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
base_rf.fit(X_train, y_train)
print("Baseline accuracy:", base_rf.score(X_test, y_test))

Baseline accuracy: 0.956140350877193


In [ ]:
Baseline first ----> Always check default RF performance before tuning. Often it's already very good.
n_jobs=-1 ----> RF parallelizes across trees. Use all cores. Makes 500 trees as fast as ~50.

If baseline RF accuracy is already 95%+, tuning might only improve by 0.5-1%. For interviews, showing you checked baseline first is impressive.

Step 3 — RandomizedSearchCV

RF has many params and large ranges. RandomizedSearchCV is the right tool here — it samples combinations instead of trying all of them.

In [2]:
param_dist = {
    'n_estimators':      [100, 200, 300, 500],
    'max_depth':          [None, 5, 10, 20, 30],
    'max_features':       ['sqrt', 'log2', 0.5],
    'min_samples_split':  [2, 5, 10],
    'min_samples_leaf':   [1, 2, 4],
    'bootstrap':          [True, False]
}
# Total combos: 4x5x3x3x3x2 = 1080. Too many for Grid.
# RandomizedSearch with n_iter=30 → only 30 fits x 5 = 150 model trains

rf = RandomForestClassifier(random_state=42, n_jobs=-1)

rand = RandomizedSearchCV(
    rf, param_dist,
    n_iter=30,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

rand.fit(X_train, y_train)

print(rand.best_params_)
print("Best ROC-AUC:", round(rand.best_score_, 4))

Fitting 5 folds for each of 30 candidates, totalling 150 fits
{'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': None, 'bootstrap': False}
Best ROC-AUC: 0.991


In [ ]:
n_iter=30 ---> Try 30 random combinations. For RF, this is usually enough to find near-optimal params
max_features=0.5 ---> Use 50% of features at each split — between sqrt and all features
bootstrap=False ---> Use whole dataset for each tree (pasting instead of bagging). Sometimes better.

After RandomizedSearch, you can do a fine-grained GridSearch around the best params found. Example: best C was 200, now search [150, 175, 200, 225, 250].